# VinAI Translate baseline trên Google Colab

Notebook chạy cùng code/config với máy local: kiểm tra dữ liệu và tokenizer, smoke test, sau đó zero-shot hoặc fine-tune LoRA/full. Trước khi chạy, chọn **Runtime → Change runtime type → GPU**. Public test không được dùng để huấn luyện hoặc chọn checkpoint.

In [ ]:
import subprocess, sys
subprocess.run(["nvidia-smi"], check=False)
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU runtime trước khi tiếp tục.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import os, subprocess, tempfile, zipfile
from google.colab import files

# Private repo: nén thư mục dự án thành một file .zip rồi chọn file đó ở đây.
# ZIP phải chứa requirements-training.txt, thư mục src/ và configs/.
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError("Hãy upload đúng một file ZIP chứa mã nguồn dự án.")

zip_name = zip_names[0]
zip_path = Path("/content") / Path(zip_name).name
zip_path.write_bytes(uploaded[zip_name])
extract_dir = Path(tempfile.mkdtemp(prefix="nlp_upload_", dir="/content"))
extract_root = extract_dir.resolve()

with zipfile.ZipFile(zip_path) as archive:
    # Chặn đường dẫn kiểu ../ để ZIP không ghi file ra ngoài thư mục tạm.
    for member in archive.infolist():
        destination = (extract_dir / member.filename).resolve()
        if not destination.is_relative_to(extract_root):
            raise ValueError(f"ZIP chứa đường dẫn không an toàn: {member.filename}")
    archive.extractall(extract_dir)

candidates = sorted({
    path.parent
    for path in extract_dir.rglob("requirements-training.txt")
    if (path.parent / "src").is_dir() and (path.parent / "configs").is_dir()
}, key=lambda path: (len(path.parts), str(path)))
if not candidates:
    raise FileNotFoundError(
        "Không tìm thấy gốc dự án trong ZIP (cần requirements-training.txt, src/ và configs/)."
    )
if len(candidates) > 1:
    print("Tìm thấy nhiều thư mục dự án; dùng thư mục ngoài cùng:", candidates[0])
PROJECT_DIR = candidates[0]
os.chdir(PROJECT_DIR)
print("Đã giải nén ZIP:", zip_path.name)
print("Project:", Path.cwd())

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-training.txt"], check=True)
print("Đã cài dependency. Nếu pip yêu cầu restart runtime, hãy restart rồi chạy lại từ cell GPU.")

In [ ]:
# Model public nên HF_TOKEN không bắt buộc; token giúp hạn mức tải ổn định hơn.
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        print("Đã nạp HF_TOKEN từ Colab Secrets.")
except Exception:
    print("Không có HF_TOKEN; tiếp tục tải model public.")

In [ ]:
# Chỉ sửa ba biến này. MODE: zero_shot | lora | full
DIRECTION = "en2vi"
MODE = "lora"
RUN_FULL = False  # đổi True sau khi smoke test thành công

if DIRECTION not in {"en2vi", "vi2en"}:
    raise ValueError("DIRECTION phải là en2vi hoặc vi2en")
if MODE not in {"zero_shot", "lora", "full"}:
    raise ValueError("MODE phải là zero_shot, lora hoặc full")
CONFIG = PROJECT_DIR / "configs" / "vinai" / f"{DIRECTION}_{MODE}.json"
DATA_DIR = PROJECT_DIR / "artifacts" / "processed_sklearn_seed67"
OUTPUT_DIR = Path("/content/drive/MyDrive/NLP_runs") / f"vinai_{DIRECTION}_{MODE}"
required = [DATA_DIR / f"segments_dev_{DIRECTION}.jsonl"]
if MODE != "zero_shot":
    required.append(DATA_DIR / f"segments_train_{DIRECTION}.jsonl")
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Thiếu dữ liệu: " + ", ".join(missing))
print("Config:", CONFIG)
print("Output bền vững trên Drive:", OUTPUT_DIR)

In [ ]:
def run_cli(command, *extra):
    args = [sys.executable, "-m", "src.vinai_baseline.cli", command,
            "--config", str(CONFIG), "--output-dir", str(OUTPUT_DIR), *map(str, extra)]
    print("RUN:", " ".join(args))
    subprocess.run(args, cwd=PROJECT_DIR, check=True)

# Audit tải tokenizer nhưng chưa tải toàn bộ model.
run_cli("audit")

In [ ]:
# Smoke test nhỏ để phát hiện lỗi môi trường/config trước full run.
COMMAND = "evaluate" if MODE == "zero_shot" else "train"
SMOKE_OUTPUT = Path("/content/vinai_smoke") / f"{DIRECTION}_{MODE}"
old_output = OUTPUT_DIR
OUTPUT_DIR = SMOKE_OUTPUT
if MODE == "zero_shot":
    run_cli(COMMAND, "--max-validation-samples", "16")
else:
    run_cli(COMMAND, "--max-train-samples", "32", "--max-validation-samples", "16")
OUTPUT_DIR = old_output
print("Smoke test hoàn thành.")

In [ ]:
# Full run chỉ bắt đầu khi bạn chủ động đặt RUN_FULL=True ở cell cấu hình.
if RUN_FULL:
    run_cli(COMMAND)
else:
    print("Chưa chạy full. Kiểm tra smoke output, đặt RUN_FULL=True rồi chạy lại cell này.")

In [ ]:
# Xem nhanh artifact; LoRA best_model là adapter và cần base model khi inference.
import json
for name in ["resolved_config.json", "reports/token_audit_summary.json",
             "reports/validation_metrics.json", "run_manifest.json"]:
    path = OUTPUT_DIR / name
    if path.exists():
        print(f"\n--- {path} ---")
        print(path.read_text(encoding="utf-8")[:4000])